# Task 4 - Data Transformation and Storage

## 1. Import packages and define paths 

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path
import duckdb
import pandas as pd

# Define project paths
project_root = Path(os.getcwd())
processed_dir = project_root / "data_outputs" / "processed"
output_db = processed_dir / "assignment1_task4.duckdb"

required_files = {
    "nger_facility_clean": processed_dir / "nger_facility_clean.csv",
    "cer_power_stations_clean": processed_dir / "cer_power_stations_clean.csv",
    "facility_geocoded": processed_dir / "facility_geocoded.csv",
    "abs_state_measures_clean": processed_dir / "abs_state_measures_clean.csv",
    "abs_measure_coverage": processed_dir / "abs_measure_coverage.csv",
    "integrated_state_year": processed_dir / "integrated_state_year.csv",
}

print("Project root:", project_root)
print("Processed dir:", processed_dir)
print("Output database:", output_db)


Project root: /Users/leitao/Documents/Semester3/5339/Assignment1
Processed dir: /Users/leitao/Documents/Semester3/5339/Assignment1/data_outputs/processed
Output database: /Users/leitao/Documents/Semester3/5339/Assignment1/data_outputs/processed/assignment1_task4.duckdb


## 2. Validate required inputs  


In [3]:
# Check whether all required files exist
missing_files = [str(path) for path in required_files.values() if not path.exists()]

if missing_files:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing_files))

pd.DataFrame({
    "dataset_name": list(required_files.keys()),
    "file_path": [str(path) for path in required_files.values()],
    "exists": [path.exists() for path in required_files.values()]
})


,dataset_name,file_path,exists
0,nger_facility_clean,/Users/leitao/Documents/Semester3/5339/Assignm...,True
1,cer_power_stations_clean,/Users/leitao/Documents/Semester3/5339/Assignm...,True
2,facility_geocoded,/Users/leitao/Documents/Semester3/5339/Assignm...,True
3,abs_state_measures_clean,/Users/leitao/Documents/Semester3/5339/Assignm...,True
4,abs_measure_coverage,/Users/leitao/Documents/Semester3/5339/Assignm...,True
5,integrated_state_year,/Users/leitao/Documents/Semester3/5339/Assignm...,True


## 3. Create DuckDB connection and load spatial extension

In [6]:
con = duckdb.connect(str(output_db))

# Try to install and load the spatial extension
spatial_loaded = False

try:
    con.execute("INSTALL spatial;")
except Exception as e:
    print("INSTALL spatial skipped:", e)

try:
    con.execute("LOAD spatial;")
    spatial_loaded = True
    print("Spatial extension loaded successfully.")
except Exception as e:
    print("LOAD spatial skipped:", e)
    print("The database can still be created, but geometry column creation will fall back.")

spatial_loaded


Spatial extension loaded successfully.


True

## 4. Create staging views from processed CSV files

In [7]:
# Create staging views directly from CSV files
for view_name, path in required_files.items():
    con.execute(
        f"""
        CREATE OR REPLACE VIEW stg_{view_name} AS
        SELECT *
        FROM read_csv_auto('{path.as_posix()}', SAMPLE_SIZE=-1, HEADER=TRUE);
        """
    )

con.execute("SHOW TABLES").fetchdf()


,name
0,stg_abs_measure_coverage
1,stg_abs_state_measures_clean
2,stg_cer_power_stations_clean
3,stg_facility_geocoded
4,stg_integrated_state_year
5,stg_nger_facility_clean


## 5. Build dimension tables

In [8]:
# Dimension: state
con.execute("""
CREATE OR REPLACE TABLE dim_state AS
SELECT
    row_number() OVER (ORDER BY state) AS state_id,
    state
FROM (
    SELECT DISTINCT state FROM stg_nger_facility_clean WHERE state IS NOT NULL
    UNION
    SELECT DISTINCT state FROM stg_cer_power_stations_clean WHERE state IS NOT NULL
    UNION
    SELECT DISTINCT state FROM stg_abs_state_measures_clean WHERE state IS NOT NULL
    UNION
    SELECT DISTINCT state FROM stg_integrated_state_year WHERE state IS NOT NULL
) t
ORDER BY state;
""")

# Dimension: reporting period
con.execute("""
CREATE OR REPLACE TABLE dim_reporting_period AS
SELECT
    row_number() OVER (ORDER BY reporting_year_start, reporting_year_end, reporting_period) AS reporting_period_id,
    reporting_period,
    reporting_year_start,
    reporting_year_end,
    abs_reference_year
FROM (
    SELECT DISTINCT reporting_period, reporting_year_start, reporting_year_end, abs_reference_year
    FROM stg_nger_facility_clean
    UNION
    SELECT DISTINCT reporting_period, reporting_year_start, reporting_year_end, abs_reference_year
    FROM stg_facility_geocoded
    UNION
    SELECT DISTINCT reporting_period, reporting_year_start, reporting_year_end, abs_reference_year
    FROM stg_integrated_state_year
) t
ORDER BY reporting_year_start, reporting_year_end, reporting_period;
""")

# Dimension: ABS measure
con.execute("""
CREATE OR REPLACE TABLE dim_abs_measure AS
SELECT
    row_number() OVER (ORDER BY m.measure_code) AS measure_id,
    m.measure_code,
    m.measure_name,
    m.parent_description,
    m.description,
    c.first_available_year,
    c.last_available_year,
    c.available_year_count,
    c.state_count,
    c.expected_year_count,
    c.coverage_ratio
FROM (
    SELECT DISTINCT measure_code, measure_name, parent_description, description
    FROM stg_abs_state_measures_clean
) m
LEFT JOIN stg_abs_measure_coverage c
    ON m.measure_code = c.measure_code;
""")

# Dimension: reporting entity
con.execute("""
CREATE OR REPLACE TABLE dim_reporting_entity AS
WITH ranked_entities AS (
    SELECT
        reporting_entity_key,
        reporting_entity,
        row_number() OVER (
            PARTITION BY reporting_entity_key
            ORDER BY reporting_entity
        ) AS rn
    FROM stg_nger_facility_clean
    WHERE reporting_entity IS NOT NULL
      AND reporting_entity_key IS NOT NULL
)
SELECT
    row_number() OVER (ORDER BY reporting_entity_key) AS reporting_entity_id,
    reporting_entity_key,
    reporting_entity
FROM ranked_entities
WHERE rn = 1;
""")

# Dimension: facility
con.execute("""
CREATE OR REPLACE TABLE dim_facility AS
WITH ranked_facilities AS (
    SELECT
        facility_name_key,
        state,
        facility_name_canonical,
        facility_name_core,
        facility_name,
        type,
        grid_connected,
        grid,
        primary_fuel,
        primary_fuel_family,
        row_number() OVER (
            PARTITION BY facility_name_key, state
            ORDER BY reporting_year_end DESC, facility_name
        ) AS rn
    FROM stg_nger_facility_clean
    WHERE facility_name IS NOT NULL
      AND facility_name_key IS NOT NULL
      AND state IS NOT NULL
)
SELECT
    row_number() OVER (ORDER BY facility_name_key, state) AS facility_id,
    facility_name_key,
    facility_name_canonical,
    facility_name_core,
    facility_name,
    state,
    type,
    grid_connected,
    grid,
    primary_fuel,
    primary_fuel_family
FROM ranked_facilities
WHERE rn = 1;
""")

# Dimension: renewable power station
con.execute("""
CREATE OR REPLACE TABLE dim_power_station AS
SELECT
    row_number() OVER (ORDER BY accreditation_code) AS power_station_id,
    accreditation_code,
    power_station_name,
    power_station_name_key,
    power_station_name_canonical,
    power_station_name_core,
    state,
    postcode,
    installed_capacity,
    fuel_source_s,
    fuel_family,
    accreditation_start_date,
    accreditation_start_year,
    suspension_status,
    baseline_mwh,
    comment,
    source_file
FROM stg_cer_power_stations_clean;
""")

con.execute("SHOW TABLES").fetchdf()


,name
0,dim_abs_measure
1,dim_facility
2,dim_power_station
3,dim_reporting_entity
4,dim_reporting_period
5,dim_state
6,stg_abs_measure_coverage
7,stg_abs_state_measures_clean
8,stg_cer_power_stations_clean
9,stg_facility_geocoded


## 6. Build bridge and fact tables


In [9]:
# Bridge table between NGER facilities and CER power stations
con.execute("""
CREATE OR REPLACE TABLE bridge_facility_power_station AS
SELECT DISTINCT
    f.facility_id,
    p.power_station_id,
    g.facility_match_method,
    g.facility_match_status,
    g.facility_match_found,
    g.matched_is_renewable_power_station
FROM stg_facility_geocoded g
JOIN dim_facility f
    ON g.facility_name_key = f.facility_name_key
   AND g.state = f.state
JOIN dim_power_station p
    ON g.matched_accreditation_code = p.accreditation_code
WHERE g.matched_accreditation_code IS NOT NULL;
""")

# Fact table: NGER facility-year observations
con.execute("""
CREATE OR REPLACE TABLE fact_nger_facility_year AS
SELECT
    n.nger_row_id,
    f.facility_id,
    e.reporting_entity_id,
    s.state_id,
    rp.reporting_period_id,
    n.electricity_production_gj,
    n.electricity_production_mwh,
    n.total_scope_1_emissions_t_co2_e,
    n.total_scope_2_emissions_t_co2_e,
    n.total_emissions_t_co2_e,
    n.emission_intensity_t_co2_e_mwh,
    n.important_notes,
    n.source_file
FROM stg_nger_facility_clean n
JOIN dim_facility f
    ON n.facility_name_key = f.facility_name_key
   AND n.state = f.state
JOIN dim_reporting_entity e
    ON n.reporting_entity_key = e.reporting_entity_key
JOIN dim_state s
    ON n.state = s.state
JOIN dim_reporting_period rp
    ON n.reporting_period = rp.reporting_period
   AND n.reporting_year_start = rp.reporting_year_start
   AND n.reporting_year_end = rp.reporting_year_end
   AND n.abs_reference_year = rp.abs_reference_year;
""")

# Fact table: facility location
if spatial_loaded:
    con.execute("""
    CREATE OR REPLACE TABLE fact_facility_location AS
    WITH facility_geo AS (
        SELECT
            f.facility_id,
            s.state_id,
            avg(g.latitude) AS latitude,
            avg(g.longitude) AS longitude,
            max(g.facility_match_method) AS facility_match_method,
            max(g.facility_match_status) AS facility_match_status,
            max(g.matched_accreditation_code) AS matched_accreditation_code,
            max(g.matched_is_renewable_power_station) AS matched_is_renewable_power_station
        FROM stg_facility_geocoded g
        JOIN dim_facility f
            ON g.facility_name_key = f.facility_name_key
           AND g.state = f.state
        JOIN dim_state s
            ON g.state = s.state
        GROUP BY f.facility_id, s.state_id
    )
    SELECT
        facility_id,
        state_id,
        latitude,
        longitude,
        CASE
            WHEN longitude IS NOT NULL AND latitude IS NOT NULL THEN ST_Point(longitude, latitude)
            ELSE NULL
        END AS geom_point,
        facility_match_method,
        facility_match_status,
        matched_accreditation_code,
        matched_is_renewable_power_station
    FROM facility_geo;
    """)
else:
    con.execute("""
    CREATE OR REPLACE TABLE fact_facility_location AS
    SELECT
        f.facility_id,
        s.state_id,
        avg(g.latitude) AS latitude,
        avg(g.longitude) AS longitude,
        max(g.facility_match_method) AS facility_match_method,
        max(g.facility_match_status) AS facility_match_status,
        max(g.matched_accreditation_code) AS matched_accreditation_code,
        max(g.matched_is_renewable_power_station) AS matched_is_renewable_power_station
    FROM stg_facility_geocoded g
    JOIN dim_facility f
        ON g.facility_name_key = f.facility_name_key
       AND g.state = f.state
    JOIN dim_state s
        ON g.state = s.state
    GROUP BY f.facility_id, s.state_id;
    """)

# Fact table: ABS state-year-measure values
con.execute("""
CREATE OR REPLACE TABLE fact_abs_state_measure AS
SELECT
    m.measure_id,
    s.state_id,
    asm.abs_reference_year,
    asm.value,
    asm.source_file
FROM stg_abs_state_measures_clean asm
JOIN dim_abs_measure m
    ON asm.measure_code = m.measure_code
JOIN dim_state s
    ON asm.state = s.state;
""")

# Fact table: integrated state-year summary
con.execute("""
CREATE OR REPLACE TABLE fact_state_year_summary AS
SELECT
    s.state_id,
    rp.reporting_period_id,
    i.facility_count,
    i.reporting_entity_count,
    i.electricity_production_mwh,
    i.electricity_production_gj,
    i.total_scope_1_emissions_t_co2_e,
    i.total_scope_2_emissions_t_co2_e,
    i.total_emissions_t_co2_e,
    i.emissions_intensity_t_co2_e_per_mwh,
    i.cumulative_renewable_power_station_count,
    i.cumulative_renewable_capacity_mw,
    i.business_entries_count,
    i.business_exits_count,
    i.employee_income_earners_count,
    i.mean_employee_income_aud,
    i.median_employee_income_aud,
    i.total_businesses_count,
    i.total_employee_income_million_aud,
    i.utilities_businesses_count,
    i.renewable_capacity_mw_per_1000_businesses,
    i.electricity_mwh_per_income_earner,
    i.emissions_t_co2_e_per_income_earner
FROM stg_integrated_state_year i
JOIN dim_state s
    ON i.state = s.state
JOIN dim_reporting_period rp
    ON i.reporting_period = rp.reporting_period
   AND i.reporting_year_start = rp.reporting_year_start
   AND i.reporting_year_end = rp.reporting_year_end
   AND i.abs_reference_year = rp.abs_reference_year;
""")

con.execute("SHOW TABLES").fetchdf()

,name
0,bridge_facility_power_station
1,dim_abs_measure
2,dim_facility
3,dim_power_station
4,dim_reporting_entity
5,dim_reporting_period
6,dim_state
7,fact_abs_state_measure
8,fact_facility_location
9,fact_nger_facility_year


## 7. Create analysis views


In [10]:
# View for dashboard-style state-year analysis
con.execute("""
CREATE OR REPLACE VIEW vw_state_year_dashboard AS
SELECT
    s.state,
    rp.reporting_period,
    rp.reporting_year_start,
    rp.reporting_year_end,
    rp.abs_reference_year,
    fsys.facility_count,
    fsys.reporting_entity_count,
    fsys.electricity_production_mwh,
    fsys.total_emissions_t_co2_e,
    fsys.emissions_intensity_t_co2_e_per_mwh,
    fsys.cumulative_renewable_capacity_mw,
    fsys.business_entries_count,
    fsys.business_exits_count,
    fsys.mean_employee_income_aud,
    fsys.total_businesses_count
FROM fact_state_year_summary fsys
JOIN dim_state s ON fsys.state_id = s.state_id
JOIN dim_reporting_period rp ON fsys.reporting_period_id = rp.reporting_period_id;
""")

# View for map-ready facility output
con.execute("""
CREATE OR REPLACE VIEW vw_facility_emissions_map AS
SELECT
    f.facility_name,
    s.state,
    rp.reporting_period,
    n.electricity_production_mwh,
    n.total_emissions_t_co2_e,
    l.latitude,
    l.longitude,
    l.matched_is_renewable_power_station
FROM fact_nger_facility_year n
JOIN dim_facility f ON n.facility_id = f.facility_id
JOIN dim_state s ON n.state_id = s.state_id
JOIN dim_reporting_period rp ON n.reporting_period_id = rp.reporting_period_id
LEFT JOIN fact_facility_location l
    ON n.facility_id = l.facility_id
   AND n.state_id = l.state_id;
""")

con.execute("SHOW TABLES").fetchdf()


,name
0,bridge_facility_power_station
1,dim_abs_measure
2,dim_facility
3,dim_power_station
4,dim_reporting_entity
5,dim_reporting_period
6,dim_state
7,fact_abs_state_measure
8,fact_facility_location
9,fact_nger_facility_year


## 8. Run validation checks

In [11]:
# Basic row-count validation
validation_results = pd.DataFrame({
    "table_name": [
        "dim_state",
        "dim_reporting_period",
        "dim_abs_measure",
        "dim_reporting_entity",
        "dim_facility",
        "dim_power_station",
        "bridge_facility_power_station",
        "fact_nger_facility_year",
        "fact_facility_location",
        "fact_abs_state_measure",
        "fact_state_year_summary",
    ],
    "row_count": [
        con.execute("SELECT COUNT(*) FROM dim_state").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM dim_reporting_period").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM dim_abs_measure").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM dim_reporting_entity").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM dim_facility").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM dim_power_station").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM bridge_facility_power_station").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM fact_nger_facility_year").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM fact_facility_location").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM fact_abs_state_measure").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM fact_state_year_summary").fetchone()[0],
    ]
})

validation_results


,table_name,row_count
0,dim_state,8
1,dim_reporting_period,10
2,dim_abs_measure,8
3,dim_reporting_entity,260
4,dim_facility,838
5,dim_power_station,3432
6,bridge_facility_power_station,251
7,fact_nger_facility_year,4862
8,fact_facility_location,838
9,fact_abs_state_measure,304


## 9. Preview query outputs


In [12]:
# Preview state-year dashboard view
con.execute("""
SELECT *
FROM vw_state_year_dashboard
ORDER BY state, reporting_year_start
LIMIT 20;
""").fetchdf()


,state,reporting_period,reporting_year_start,reporting_year_end,abs_reference_year,facility_count,reporting_entity_count,electricity_production_mwh,total_emissions_t_co2_e,emissions_intensity_t_co2_e_per_mwh,cumulative_renewable_capacity_mw,business_entries_count,business_exits_count,mean_employee_income_aud,total_businesses_count
0,ACT,2014-15,2014,2015,2015,3,3,98106.0,1708.0,0.017410,22.4367,NaN,NaN,NaN,NaN
1,ACT,2015-16,2015,2016,2016,3,2,63256.0,1788.0,0.028266,49.9140,NaN,NaN,NaN,NaN
2,ACT,2016-17,2016,2017,2017,3,2,63949.0,1956.0,0.030587,49.9140,NaN,NaN,NaN,NaN
3,ACT,2017-18,2017,2018,2018,3,2,73003.0,2355.0,0.032259,50.7568,NaN,NaN,72998.0,NaN
4,ACT,2018-19,2018,2019,2019,5,3,93555.0,2061.0,0.022030,51.3561,NaN,NaN,75131.0,NaN
5,ACT,2019-20,2019,2020,2020,5,4,87670.0,1575.0,0.017965,59.4687,NaN,NaN,77876.0,29724.0
6,ACT,2020-21,2020,2021,2021,5,4,90253.0,2979.0,0.033007,61.1264,5560.0,3839.0,80960.0,31490.0
7,ACT,2021-22,2021,2022,2022,5,4,84427.0,3786.0,0.044843,61.7141,6591.0,4438.0,83505.0,33813.0
8,ACT,2022-23,2022,2023,2023,9,5,246566.0,5287.0,0.021443,63.3304,6207.0,4962.0,NaN,35086.0
9,ACT,2023-24,2023,2024,2024,9,5,224556.0,5621.0,0.025032,65.3910,6501.0,5138.0,NaN,36316.0


In [13]:
# Preview facility map view
con.execute("""
SELECT *
FROM vw_facility_emissions_map
WHERE latitude IS NOT NULL AND longitude IS NOT NULL
ORDER BY total_emissions_t_co2_e DESC
LIMIT 20;
""").fetchdf()


,facility_name,state,reporting_period,electricity_production_mwh,total_emissions_t_co2_e,latitude,longitude,matched_is_renewable_power_station
0,Loy Yang Power Station and Mine,VIC,2017-18,16951912.0,20107115.0,-38.253707,146.575663,False
1,Loy Yang Power Station and Mine,VIC,2020-21,16397733.0,19373698.0,-38.253707,146.575663,False
2,Loy Yang Power Station and Mine,VIC,2016-17,15885288.0,18963597.0,-38.253707,146.575663,False
3,Loy Yang Power Station and Mine,VIC,2018-19,15959544.0,18798519.0,-38.253707,146.575663,False
4,Loy Yang Power Station and Mine,VIC,2014-15,16226412.0,18777224.0,-38.253707,146.575663,False
5,Loy Yang Power Station and Mine,VIC,2023-24,15689642.0,18723707.0,-38.253707,146.575663,False
6,Loy Yang Power Station and Mine,VIC,2015-16,15715547.0,18401416.0,-38.253707,146.575663,False
7,Loy Yang Power Station and Mine,VIC,2021-22,15143831.0,17948332.0,-38.253707,146.575663,False
8,Loy Yang Power Station and Mine,VIC,2019-20,14663279.0,16926277.0,-38.253707,146.575663,False
9,Loy Yang Power Station and Mine,VIC,2022-23,13929522.0,16360431.0,-38.253707,146.575663,False


## 10. Close connection

In [14]:
con.close()
print(f"Task 4 database created successfully at: {output_db}")


Task 4 database created successfully at: /Users/leitao/Documents/Semester3/5339/Assignment1/data_outputs/processed/assignment1_task4.duckdb
